# 距離はしごを最初から最後まで自分でつなぐ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/distance/distance_ladder.ipynb)

[sn_ia_hubble.ipynb](sn_ia_hubble.ipynb) では、超新星の絶対等級 $M_B$ の較正に、SH0ES が求めた母銀河のセファイド距離(`CEPH_DIST`)をそのまま使いました。
このノートブックでは、そこを自分でつなぎ、**Gaia の視差から $H_0$ まで、途中で他人の距離を借りずに** たどります。

```
段1  Gaia の視差 ──→ 天の川銀河のセファイド ──→ LMC までの距離          (数百 pc → 50 kpc)
段2  LMC のセファイド(HST)──→ HST の近赤外での周期−光度関係のゼロ点
段3  SN 母銀河のセファイド(HST)──→ 37 個の母銀河までの距離            (7〜40 Mpc)
段4  母銀河の Ia 型超新星 ──→ 超新星の絶対等級 M_B
段5  ハッブル流の超新星(0.023 < z < 0.15)──→ H₀                    (100〜600 Mpc)
```

外から借りるのは「測った量」(視差、周期、等級、赤方偏移)だけで、**距離はすべて自分で計算** します。
答え合わせ用に、LMC の食連星の距離、NGC 4258 のメーザーの距離、SH0ES の母銀河の距離を途中で比べます。

## なぜ LMC を 2 回使うのか

Gaia の等級(G, BP, RP)と、遠くの銀河のセファイドを測った HST の等級(F555W, F814W, F160W)は、フィルターも装置も違います。
周期−光度関係の「ゼロ点」(=セファイドの本当の明るさ)をフィルターをまたいで持ち運ぶのは難しいので、

- 段1 で Gaia の世界の中で **LMC までの距離** を決め、
- 段2 で、HST で測った LMC のセファイドにその距離を当てはめて、**HST の世界でのゼロ点** を決める

というように、LMC を「2 つの世界をつなぐ橋」にします。距離はフィルターによらないので、これなら持ち運べます。

## 使うデータ

- **Gaia DR3**(ESA, オープン): `gaiadr3.vari_cepheid`、`gaiadr3.gaia_source`
- **SH0ES**(Riess et al. 2022)の HST セファイド測光(Table 2): LMC・NGC 4258・SN 母銀河の約 3000 個のセファイドの周期、色、F160W 等級、金属量
- **Pantheon+**(Scolnic et al. 2022, Brout et al. 2022)の超新星の等級と赤方偏移

SH0ES と Pantheon+ のデータは https://github.com/PantheonPlusSH0ES/DataRelease にあります。リポジトリにライセンスの明記はなく、研究用に公開されたデータなので、使うときは論文を引用します。

## 0. 準備

In [ ]:
# Colab には astroquery が入っていないのでインストールする
!pip install -q astroquery

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from urllib.request import urlopen

from astroquery.gaia import Gaia

Gaia.ROW_LIMIT = -1

RELEASE = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
SH0ES_TABLE_URL = RELEASE + "SH0ES_Data/table2.tex"
PANTHEON_URL = RELEASE + "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"

C_KMS = 299_792.458  # 光速 [km/s]

# 金属量による周期−光度関係のずれ [mag/dex](Riess et al. 2022 の値)。
# 金属量 [O/H] が低いセファイドほど暗い。0 にすると補正なし
GAMMA = -0.217
OH_LMC = -0.29  # LMC の [O/H](太陽 = 0。SH0ES の表の値)


def distance_from_modulus(mu):
    # 距離指数 m - M → 距離 [pc]
    return 10.0 ** ((mu + 5.0) / 5.0)

## 段1. Gaia の視差で LMC までの距離を決める

[cepheid.ipynb](cepheid.ipynb) と同じことを、短くまとめて繰り返します。

- Gaia DR3 の古典的セファイド(基本振動)を取り、減光の影響を打ち消すヴェーゼンハイト等級 $W_G = G - 1.90\,(G_{BP} - G_{RP})$ を使う
- 視差のよい天の川銀河のセファイドで周期−光度関係のゼロ点を決める
- LMC のセファイドの見かけの関係と比べて、LMC の距離指数を出す

1 つだけ追加するのが **金属量の補正** です。LMC は天の川銀河(太陽付近)より金属量が低く、同じ周期でもセファイドが少し暗くなります。
補正しないと、その分だけ LMC を遠くに見積もってしまいます。ここでは天の川銀河のセファイドを [O/H] ≈ 0 とみなし、

$$
\mu_{\mathrm{LMC}} = a_{\mathrm{LMC}}^{\mathrm{app}} - a_{\mathrm{MW}} - \gamma\,[\mathrm{O/H}]_{\mathrm{LMC}}
$$

とします。$\gamma$ は SH0ES が HST の近赤外で求めた値で、Gaia のバンドでも大きくは違わないと仮定しています(ここは粗い仮定です)。

In [ ]:
LAMBDA_W = 1.90  # Gaia バンドのヴェーゼンハイト係数

query = '''
SELECT c.source_id, c.pf, c.int_average_g, c.int_average_bp, c.int_average_rp,
       g.ra, g.dec, g.parallax, g.parallax_error, g.ruwe
FROM gaiadr3.vari_cepheid AS c
JOIN gaiadr3.gaia_source AS g USING (source_id)
WHERE c.type_best_classification = 'DCEP'
  AND c.mode_best_classification = 'FUNDAMENTAL'
  AND c.pf IS NOT NULL
  AND c.int_average_g IS NOT NULL
  AND c.int_average_bp IS NOT NULL
  AND c.int_average_rp IS NOT NULL
'''
ceph = Gaia.launch_job_async(query).get_results().to_pandas()
ceph.columns = ceph.columns.str.lower()
ceph["logp"] = np.log10(ceph["pf"])
ceph["w"] = ceph["int_average_g"] - LAMBDA_W * (ceph["int_average_bp"] - ceph["int_average_rp"])

LMC_CENTER = SkyCoord(80.894 * u.deg, -69.756 * u.deg)
SMC_CENTER = SkyCoord(13.187 * u.deg, -72.829 * u.deg)
coords = SkyCoord(ceph["ra"].to_numpy() * u.deg, ceph["dec"].to_numpy() * u.deg)
in_lmc = coords.separation(LMC_CENTER).deg < 8.0
in_smc = coords.separation(SMC_CENTER).deg < 4.0

gaia_lmc = ceph[in_lmc].copy()

PLX_ZP = -0.017  # Gaia DR3 の視差ゼロ点 [mas](Lindegren et al. 2021)
gaia_mw = ceph[~in_lmc & ~in_smc].copy()
gaia_mw["plx"] = gaia_mw["parallax"] - PLX_ZP
gaia_mw = gaia_mw[(gaia_mw["plx"] / gaia_mw["parallax_error"] > 10) & (gaia_mw["ruwe"] < 1.4)].copy()
gaia_mw["abs_w"] = gaia_mw["w"] + 5 * np.log10(gaia_mw["plx"]) - 10
gaia_mw["abs_w_err"] = 5 / np.log(10) * gaia_mw["parallax_error"] / gaia_mw["plx"]
print(f"LMC: {len(gaia_lmc)} 個, 視差のよい天の川銀河のセファイド: {len(gaia_mw)} 個")

In [ ]:
INTRINSIC_SCATTER = 0.10  # 周期-光度関係そのものの幅 [mag]


def fit_line(x, y, yerr, n_iter=5, clip=3.0):
    # 重み付き直線当てはめ + 外れ値除去。(傾き, 切片), 共分散, 使った点のマスク を返す
    sig = np.sqrt(yerr ** 2 + INTRINSIC_SCATTER ** 2)
    keep = np.ones(len(x), dtype=bool)
    for _ in range(n_iter):
        coef, cov = np.polyfit(x[keep], y[keep], 1, w=1 / sig[keep], cov="unscaled")
        resid = y - np.polyval(coef, x)
        keep = np.abs(resid) < clip * sig
    return coef, cov, keep


# LMC: 傾きと見かけの切片
(b_lmc, a_lmc_app), _, _ = fit_line(gaia_lmc["logp"].to_numpy(), gaia_lmc["w"].to_numpy(), np.zeros(len(gaia_lmc)))

# 天の川銀河: 外れ値を外したうえで、傾きを LMC の値に固定して切片を求める
x_mw = gaia_mw["logp"].to_numpy()
y_mw = gaia_mw["abs_w"].to_numpy()
_, _, keep_mw = fit_line(x_mw, y_mw, gaia_mw["abs_w_err"].to_numpy())
sig_mw = np.sqrt(gaia_mw["abs_w_err"].to_numpy() ** 2 + INTRINSIC_SCATTER ** 2)
wts = 1 / sig_mw[keep_mw] ** 2
a_mw = np.sum(wts * (y_mw[keep_mw] - b_lmc * x_mw[keep_mw])) / np.sum(wts)
a_mw_err = 1 / np.sqrt(np.sum(wts))

mu_lmc = a_lmc_app - a_mw - GAMMA * OH_LMC
mu_lmc_err = a_mw_err
print(f"金属量の補正: {-GAMMA * OH_LMC:+.3f} mag")
print(f"LMC の距離指数(自分の値): {mu_lmc:.3f} ± {mu_lmc_err:.3f}  →  {distance_from_modulus(mu_lmc) / 1000:.1f} kpc")
print(f"食連星による値(Pietrzyński et al. 2019): 18.477 ± 0.026  →  49.6 kpc")

ここで決めた `mu_lmc` が、このあとのすべての距離の土台になります。
誤差 `mu_lmc_err` は、視差の統計誤差から来る切片の誤差だけです。視差のゼロ点や金属量の補正の不確かさ(それぞれ数百分の 1 等級)は入っていません。

## 段2. HST で測った LMC のセファイドで、近赤外の周期−光度関係を較正する

SH0ES の論文の Table 2 には、HST で測ったセファイドが銀河ごとに並んでいます。主な列:

| 列 | 意味 |
| --- | --- |
| `field` | 銀河(LMC、NGC 4258 = `N4258`、SN の母銀河など) |
| `P` | 周期 [日] |
| `VI` | F555W − F814W(可視の色) |
| `H` | F160W(近赤外 1.6 μm)の等級 |
| `sH` | 下のヴェーゼンハイト等級の誤差(周期−光度関係そのものの幅も含む) |
| `OH` | そのセファイドの位置の金属量 [O/H](太陽 = 0) |
| `note` | `HST` なら HST で測光、`GRND` なら地上望遠鏡で測光 |

近赤外は塵による減光が小さく、遠くの混み合った銀河の中でもセファイドを測りやすいので、SH0ES は近赤外のヴェーゼンハイト等級を使います。

$$
m_H^W = m_{F160W} - 0.386\,(m_{F555W} - m_{F814W})
$$

LMC のセファイドには地上望遠鏡で測ったものも含まれますが、遠くの銀河と同じ装置で比べたいので、HST で測ったものだけを使います。

In [ ]:
def read_shoes_table(url):
    # LaTeX の deluxetable の行(... & ... \\)を読む
    rows = []
    for line in urlopen(url).read().decode().splitlines():
        line = line.strip()
        if "&" in line and line.endswith("\\\\") and "colhead" not in line:
            rows.append([s.strip() for s in line[:-2].split("&")])
    t = pd.DataFrame(rows, columns=["field", "ra", "dec", "id", "P", "VI", "sVI", "H", "sH", "OH", "note"])
    for c in ["ra", "dec", "P", "VI", "sVI", "H", "sH", "OH"]:
        t[c] = t[c].astype(float)
    return t


shoes = read_shoes_table(SH0ES_TABLE_URL)
shoes["mW"] = shoes["H"] - 0.386 * shoes["VI"]
shoes["x"] = np.log10(shoes["P"]) - 1  # 周期 10 日を基準にする
print(f"{len(shoes)} 個のセファイド, {shoes['field'].nunique()} 個の銀河")
shoes.groupby("field").size().sort_values(ascending=False).head(10)

In [ ]:
lmc = shoes[(shoes["field"] == "LMC") & (shoes["note"] == "HST")]
coef = np.polyfit(lmc["x"], lmc["mW"], 1)
print(f"HST の LMC セファイド: {len(lmc)} 個, 傾き {coef[0]:.3f}, ばらつき {np.std(lmc['mW'] - np.polyval(coef, lmc['x'])):.3f} mag")

xx = np.linspace(-0.4, 1.0, 50)
plt.figure(figsize=(7, 5))
plt.errorbar(lmc["x"] + 1, lmc["mW"], yerr=lmc["sH"], fmt="o", ms=3, label="LMC Cepheids (HST)")
plt.plot(xx + 1, np.polyval(coef, xx), c="tab:red")
plt.gca().invert_yaxis()
plt.xlabel("log10 P [day]")
plt.ylabel("m_H^W [mag]")
plt.legend()
plt.show()

近赤外のヴェーゼンハイト等級だと、ばらつきは 0.09 mag 程度(距離にして 4%)しかありません。Gaia の可視の $W_G$ よりずっと細い関係です。

## 段3. LMC・NGC 4258・SN 母銀河のセファイドをまとめて当てはめる

銀河 $j$ のセファイド $i$ について、

$$
m_{H,ij}^W = \mu_j + M_H^W + b\,(\log_{10} P_{ij} - 1) + \gamma\,[\mathrm{O/H}]_{ij}
$$

とおきます。$M_H^W$(周期 10 日のセファイドの絶対等級)と傾き $b$ はすべての銀河で共通、$\mu_j$ は銀河ごとの距離指数です。
このままだと「$\mu_j$ を全部 1 増やして $M_H^W$ を 1 減らす」ことができて決まらないので、**LMC の $\mu$ だけは段1 で自分で出した値に固定** します。
すると LMC のセファイドで $M_H^W$ が決まり、ほかの銀河の $\mu_j$ が決まります。

未知数は直線的に入っているので、重み付き最小二乗法で一度に解けます(外れ値は 3.3σ で除く。SH0ES と同じ基準)。

NGC 4258 は超新星の母銀河ではありませんが、**水メーザーの運動から幾何学的に距離が分かっている** 銀河です。
ここでは NGC 4258 の距離を使わずに求めて、答え合わせに使います。

In [ ]:
pantheon = pd.read_csv(PANTHEON_URL, sep=r"\s+")

# 較正用超新星(セファイドのいる銀河の超新星)。複数のプロジェクトの測定は平均する
cal = (
    pantheon[pantheon["IS_CALIBRATOR"] == 1]
    .groupby("CID")
    .agg(ra=("RA", "mean"), dec=("DEC", "mean"), m_b=("m_b_corr", "mean"),
         m_b_err=("m_b_corr_err_DIAG", "mean"), mu_shoes=("CEPH_DIST", "mean"))
    .reset_index()
)
cal = cal[cal["CID"] != "2005df_ANU"]  # 2005df の別の解析なので重複を除く

# 超新星の位置から、いちばん近いセファイドの銀河を母銀河とする
fields = shoes[shoes["field"] != "LMC"].groupby("field")[["ra", "dec"]].median()
sn_coord = SkyCoord(cal["ra"].to_numpy() * u.deg, cal["dec"].to_numpy() * u.deg)
field_coord = SkyCoord(fields["ra"].to_numpy() * u.deg, fields["dec"].to_numpy() * u.deg)
idx, sep, _ = sn_coord.match_to_catalog_sky(field_coord)
cal["host"] = fields.index[idx]
cal["sep_arcmin"] = sep.arcmin
assert (cal["sep_arcmin"] < 10).all()
print(f"較正用超新星 {len(cal)} 個, 母銀河 {cal['host'].nunique()} 個")

In [ ]:
galaxies = sorted(cal["host"].unique()) + ["N4258"]  # μ を求める銀河
use = shoes[((shoes["field"] == "LMC") & (shoes["note"] == "HST")) | shoes["field"].isin(galaxies)].copy()

# y = A p。p = [μ_1, ..., μ_n, M_H^W, b]。LMC の μ と金属量の項は既知として y 側に移す
is_lmc = (use["field"] == "LMC").to_numpy()
y = use["mW"].to_numpy() - GAMMA * use["OH"].to_numpy() - np.where(is_lmc, mu_lmc, 0.0)
A = np.zeros((len(use), len(galaxies) + 2))
for k, g in enumerate(galaxies):
    A[(use["field"] == g).to_numpy(), k] = 1.0
A[:, -2] = 1.0
A[:, -1] = use["x"].to_numpy()
sig = use["sH"].to_numpy()

keep = np.ones(len(y), dtype=bool)
for _ in range(10):
    wA = A[keep] / sig[keep, None] ** 2
    cov = np.linalg.inv(A[keep].T @ wA)
    p = cov @ (wA.T @ y[keep])
    new_keep = np.abs(y - A @ p) < 3.3 * sig
    if (new_keep == keep).all():
        break
    keep = new_keep

p_err = np.sqrt(np.diag(cov))
mu_host = pd.Series(p[:-2], index=galaxies)
mu_host_err = pd.Series(p_err[:-2], index=galaxies)  # LMC の距離の誤差は含まない(あとで別に足す)
M_HW, b_HW = p[-2], p[-1]
use["keep"] = keep
print(f"使ったセファイド: {keep.sum()} / {len(y)}(LMC {is_lmc.sum()} 個を含む)")
print(f"M_H^W = {M_HW:.3f} ± {p_err[-2]:.3f}(周期 10 日), 傾き b = {b_HW:.3f} ± {p_err[-1]:.3f}")
print("SH0ES(LMC・天の川銀河・NGC 4258 の 3 つが土台): M_H^W = -5.894, b = -3.299")

In [ ]:
# いくつかの銀河で、距離指数を引いた周期−光度関係を重ねる
plt.figure(figsize=(8, 5))
for g in ["LMC", "N4258", "M101", "N5584", "N3447"]:
    s = use[(use["field"] == g) & use["keep"]]
    mu_g = mu_lmc if g == "LMC" else mu_host[g]
    plt.plot(s["x"] + 1, s["mW"] - GAMMA * s["OH"] - mu_g, ".", ms=4, alpha=0.6, label=f"{g} (μ = {mu_g:.2f})")
plt.plot(xx + 1, M_HW + b_HW * xx, c="k", label="fit")
plt.gca().invert_yaxis()
plt.xlabel("log10 P [day]")
plt.ylabel("m_H^W - γ[O/H] - μ  [mag]")
plt.legend(fontsize=8)
plt.show()

遠くの銀河のセファイドはばらつきが大きくなります(周りの星の光が混ざる「混み合い」の影響を補正した分、誤差が大きい)が、
数十〜数百個を平均するので、1 つの銀河の距離指数は 0.03〜0.1 mag の精度で決まります。

### 答え合わせ 1: NGC 4258 のメーザー距離

NGC 4258 の中心の円盤にある水メーザーの、天球上の速さと視線速度を比べると、幾何学的に距離が出ます: $7.576 \pm 0.112$ Mpc(Reid et al. 2019)。

In [ ]:
MU_N4258_MASER = 5 * np.log10(7.576e6) - 5
MU_N4258_MASER_ERR = 5 / np.log(10) * 0.112 / 7.576
mu_n, mu_n_err = mu_host["N4258"], np.hypot(mu_host_err["N4258"], mu_lmc_err)
print(f"NGC 4258(自分のはしご): μ = {mu_n:.3f} ± {mu_n_err:.3f}  →  {distance_from_modulus(mu_n) / 1e6:.2f} Mpc")
print(f"NGC 4258(メーザー):     μ = {MU_N4258_MASER:.3f} ± {MU_N4258_MASER_ERR:.3f}  →  7.58 Mpc")
print(f"差: {mu_n - MU_N4258_MASER:+.3f} mag ({(mu_n - MU_N4258_MASER) / np.hypot(mu_n_err, MU_N4258_MASER_ERR):+.1f}σ)")

差のほとんどは、段1 の LMC の距離が食連星の値とどれだけずれたかで決まります(LMC を遠くに見積もれば、NGC 4258 も同じだけ遠くなる)。

### 答え合わせ 2: SH0ES の母銀河の距離

Pantheon+ の `CEPH_DIST` は、SH0ES が 3 つの土台(LMC・天の川銀河・NGC 4258)を使って出した母銀河の距離指数です。

In [ ]:
host_tab = cal.groupby("host").agg(mu_shoes=("mu_shoes", "mean")).reset_index()
host_tab["mu_mine"] = host_tab["host"].map(mu_host)
host_tab["mu_mine_err"] = host_tab["host"].map(mu_host_err)
diff = host_tab["mu_mine"] - host_tab["mu_shoes"]
print(f"自分 − SH0ES: 平均 {diff.mean():+.3f} mag, ばらつき {diff.std():.3f} mag({len(host_tab)} 銀河)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].errorbar(host_tab["mu_shoes"], host_tab["mu_mine"], yerr=host_tab["mu_mine_err"], fmt="o", ms=4)
lim = [host_tab["mu_shoes"].min() - 0.3, host_tab["mu_shoes"].max() + 0.3]
axes[0].plot(lim, lim, c="gray", ls="--")
axes[0].set_xlabel("μ (SH0ES, CEPH_DIST)")
axes[0].set_ylabel("μ (my ladder)")
axes[1].errorbar(host_tab["mu_shoes"], diff, yerr=host_tab["mu_mine_err"], fmt="o", ms=4)
axes[1].axhline(0, c="gray", ls="--")
axes[1].axhline(diff.mean(), c="tab:red", label=f"mean {diff.mean():+.3f}")
axes[1].set_xlabel("μ (SH0ES, CEPH_DIST)")
axes[1].set_ylabel("my μ − SH0ES μ")
axes[1].legend()
plt.tight_layout()
plt.show()

銀河ごとのばらつきは数百分の 1 等級で、ほぼ全体のずれ(平均値)だけが残ります。そのずれは土台(LMC の距離)の違いから来ています。
SH0ES は共分散行列(セファイドどうしの誤差の相関)まで使って解いているので、細かい差はそこからも来ます。

## 段4. 母銀河の超新星で $M_B$ を較正する

自分で出した母銀河の距離指数 $\mu_j$ から、

$$
M_B = m_B - \mu_j
$$

同じ母銀河に超新星が 2 つ以上あるもの(NGC 1448、NGC 3147、NGC 5468、NGC 5643)もあります。

In [ ]:
cal["mu"] = cal["host"].map(mu_host)
cal["mu_err"] = cal["host"].map(mu_host_err)
cal["M_B"] = cal["m_b"] - cal["mu"]

w_cal = 1 / (cal["m_b_err"] ** 2 + cal["mu_err"] ** 2)
M_B = np.sum(w_cal * cal["M_B"]) / np.sum(w_cal)
M_B_err = cal["M_B"].std() / np.sqrt(len(cal))  # ばらつきから見積もった平均値の誤差。LMC の距離の誤差は含まない
print(f"M_B = {M_B:.3f} ± {M_B_err:.3f}(超新星 {len(cal)} 個, ばらつき {cal['M_B'].std():.2f} mag)")
print("sn_ia_hubble.ipynb(SH0ES の CEPH_DIST を使用): 約 -19.25")

plt.figure(figsize=(7, 4))
plt.errorbar(cal["mu"], cal["M_B"], yerr=np.sqrt(cal["m_b_err"] ** 2 + cal["mu_err"] ** 2), fmt="o", ms=4)
plt.axhline(M_B, c="tab:red", label=f"M_B = {M_B:.3f}")
plt.gca().invert_yaxis()
plt.xlabel("host distance modulus μ (my ladder)")
plt.ylabel("M_B [mag]")
plt.legend()
plt.show()

## 段5. ハッブル流の超新星で $H_0$ を出す

[sn_ia_hubble.ipynb](sn_ia_hubble.ipynb) と同じく、0.023 < z < 0.15 の超新星(`USED_IN_SH0ES_HF`)で、

$$
\log_{10} H_0 = \frac{M_B + 5\,a_B + 25}{5}, \qquad
a_B = \log_{10}\!\left[ cz \left( 1 + \frac{1 - q_0}{2} z - \frac{1 - q_0 - 3q_0^2 + j_0}{6} z^2 \right) \right] - 0.2\, m_B
$$

$a_B$ はハッブル流の超新星だけで決まる量(「$M_B = 0$ だったときの $\log_{10} H_0$」のようなもの)で、距離はしごとは独立です。
これに段4 の $M_B$ を入れると $H_0$ になります。

In [ ]:
Q0, J0 = -0.55, 1.0

sn = pantheon.groupby("CID").agg(
    zHD=("zHD", "mean"), m_b=("m_b_corr", "mean"), m_b_err=("m_b_corr_err_DIAG", "mean"),
    hubble_flow=("USED_IN_SH0ES_HF", "max"),
)
hf = sn[sn["hubble_flow"] == 1].copy()
z = hf["zHD"]
hf["aB"] = np.log10(C_KMS * z * (1 + (1 - Q0) / 2 * z - (1 - Q0 - 3 * Q0 ** 2 + J0) / 6 * z ** 2)) - 0.2 * hf["m_b"]
w_hf = 1 / hf["m_b_err"] ** 2
a_B = np.sum(w_hf * hf["aB"]) / np.sum(w_hf)
a_B_err = hf["aB"].std() / np.sqrt(len(hf))  # ばらつきから見積もった平均値の誤差

H0 = 10 ** ((M_B + 5 * a_B + 25) / 5)

# 誤差の内訳(log10 H0 = 0.2 M_B + a_B + 5 なので、等級の誤差は H0 の ln(10)/5 倍の相対誤差になる)
err = {
    "段1 LMC の距離(Gaia の視差)": H0 * np.log(10) / 5 * mu_lmc_err,
    "段2-4 セファイドと母銀河の超新星": H0 * np.log(10) / 5 * M_B_err,
    "段5 ハッブル流の超新星": H0 * np.log(10) * a_B_err,
}
H0_err = np.sqrt(sum(e ** 2 for e in err.values()))

print(f"ハッブル流の超新星: {len(hf)} 個, a_B = {a_B:.4f} ± {a_B_err:.4f}(SH0ES: 0.71273)")
print(f"\nH0 = {H0:.1f} ± {H0_err:.1f} km/s/Mpc")
for k, v in err.items():
    print(f"  {k}: ±{v:.1f}")
print("\nSH0ES(Riess et al. 2022): 73.0 ± 1.0,  Planck 2018(ΛCDM): 67.4 ± 0.5")

## 自分でつないだ距離はしごの全体像

各段で距離が分かった天体を、距離の対数で並べます。1 段ごとに 1〜3 桁ずつ遠くへ伸びているのが分かります。

In [ ]:
rungs = [
    ("Milky Way Cepheids\n(Gaia parallax)", 1000 / gaia_mw["plx"][keep_mw]),
    ("LMC\n(Gaia Cepheids)", [distance_from_modulus(mu_lmc)]),
    ("NGC 4258\n(HST Cepheids)", [distance_from_modulus(mu_host["N4258"])]),
    ("SN Ia hosts\n(HST Cepheids)", distance_from_modulus(host_tab["mu_mine"])),
    ("Hubble flow\n(SN Ia)", distance_from_modulus(hf["m_b"] - M_B)),
]
fig, ax = plt.subplots(figsize=(9, 4.5))
for k, (name, d) in enumerate(rungs):
    d = np.asarray(d)
    jitter = np.random.default_rng(k).uniform(-0.15, 0.15, len(d)) if len(d) > 1 else 0.0
    ax.plot(d, k + jitter, "o", ms=3 if len(d) > 1 else 9, alpha=0.6)
ax.plot(distance_from_modulus(MU_N4258_MASER), 2, "k*", ms=10, label="NGC 4258 maser (check)")
ax.plot(distance_from_modulus(18.477), 1, "kx", ms=10, label="LMC eclipsing binaries (check)")
ax.set_yticks(range(len(rungs)))
ax.set_yticklabels([r[0] for r in rungs])
ax.set_xscale("log")
ax.set_xlabel("distance [pc]")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 土台を取り替えると $H_0$ はどう変わるか

段1 の自分の LMC 距離のかわりに食連星の距離を土台にしたり、金属量の補正をやめたりして、同じ段2〜5 をやり直してみます。
LMC の距離指数が $\Delta\mu$ 変わると、母銀河の距離も $M_B$ も同じだけずれるので、$H_0 \propto 10^{-\Delta\mu/5}$ になります。

In [ ]:
def ladder_h0(mu_anchor, gamma):
    # 段2〜5 をまとめてやり直し、(H0, NGC 4258 の μ) を返す(外れ値の選び方は上の結果を使う)
    yy = use["mW"].to_numpy() - gamma * use["OH"].to_numpy() - np.where(is_lmc, mu_anchor, 0.0)
    wA = A[keep] / sig[keep, None] ** 2
    pp = np.linalg.solve(A[keep].T @ wA, wA.T @ yy[keep])
    mu_g = pd.Series(pp[:-2], index=galaxies)
    MB = np.sum(w_cal * (cal["m_b"] - cal["host"].map(mu_g))) / np.sum(w_cal)
    return 10 ** ((MB + 5 * a_B + 25) / 5), mu_g["N4258"]


cases = {
    "自分の LMC 距離 + 金属量補正(この結果)": (mu_lmc, GAMMA),
    "自分の LMC 距離, 金属量補正なし": (a_lmc_app - a_mw, 0.0),
    "食連星の LMC 距離 + 金属量補正": (18.477, GAMMA),
    "食連星の LMC 距離, 金属量補正なし": (18.477, 0.0),
}
for name, (m, g) in cases.items():
    h, mn = ladder_h0(m, g)
    print(f"{name:<28s} μ_LMC = {m:.3f}  →  μ(N4258) = {mn:.3f}(メーザー {MU_N4258_MASER:.3f}),  H0 = {h:.1f}")

金属量の補正は 2 か所に効きます。段1 では LMC を近づけ($H_0$ を上げ)、段3 では金属量の低い LMC と、金属量の高い母銀河の差を補正します。

## まとめ

- Gaia の視差 → LMC → HST のセファイド → 超新星 → $H_0$ と、距離を借りずにすべて自分でつないだ
- 途中の答え合わせ: NGC 4258 のセファイド距離はメーザーの幾何学的な距離と、母銀河の距離は SH0ES の値と、どちらも土台の差の範囲で一致する
- 食連星の LMC 距離を土台にすると $H_0 \approx 73$ になり、SH0ES の結果(73.0 ± 1.0)が再現できる
- $H_0$ の誤差のいちばん大きな部分は土台(段1)から来る。Gaia の視差で直接 LMC までの距離を決めると、視差の誤差と金属量の補正の仮定がそのまま $H_0$ に効く
- 精密な土台(食連星の LMC 距離)を使った $H_0 \approx 73$ と Planck の 67.4 との差(ハッブル・テンション)を土台だけで消すには、LMC を約 8% 遠くする必要がある([hubble_tension.ipynb](../cosmology/hubble_tension.ipynb))

## 発展メモ

- 段1 の視差の条件(`p/σ > 10`)や `PLX_ZP` を変えて、LMC の距離と $H_0$ がどう動くか
- `GAMMA` を 0 や −0.4 に変えて、金属量の補正の影響を見る
- NGC 4258 のメーザー距離を土台に加えて(`N4258` の μ を固定する)、2 つの土台で $H_0$ を出す
- SH0ES が公開している方程式の行列(`alll_...fits`)・データ(`ally_...fits`)・共分散(`allc_...fits`)を使って、誤差の相関まで入れた本来の解き方と比べる
- 母銀河の超新星を 1 つずつ抜いて(ジャックナイフ)、$M_B$ が特定の超新星に引っ張られていないか確かめる

## 参考・クレジット

- Riess et al. (2022), *A Comprehensive Measurement of the Local Value of the Hubble Constant with 1 km/s/Mpc Uncertainty from the Hubble Space Telescope and the SH0ES Team*, ApJL 934, L7
- Scolnic et al. (2022), *The Pantheon+ Analysis: The Full Data Set and Light-curve Release*, ApJ 938, 113
- Brout et al. (2022), *The Pantheon+ Analysis: Cosmological Constraints*, ApJ 938, 110
- Gaia Collaboration, Vallenari et al. (2023), *Gaia Data Release 3: Summary of the content and survey properties*, A&A 674, A1
- Ripepi et al. (2023), *Gaia Data Release 3: Specific processing and validation of all-sky RR Lyrae and Cepheid stars – The Cepheid sample*, A&A 674, A17
- Lindegren et al. (2021), *Gaia Early Data Release 3: Parallax bias versus magnitude, colour, and position*, A&A 649, A4
- Pietrzyński et al. (2019), *A distance to the Large Magellanic Cloud that is precise to one per cent*, Nature 567, 200
- Reid, Pesce & Riess (2019), *An Improved Distance to NGC 4258 and Its Implications for the Hubble Constant*, ApJL 886, L27

This work has made use of data from the European Space Agency (ESA) mission *Gaia* (https://www.cosmos.esa.int/gaia), processed by the Gaia Data Processing and Analysis Consortium (DPAC, https://www.cosmos.esa.int/web/gaia/dpac/consortium).